In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os

builder = SparkSession.builder \
    .appName("ETL Silver Layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
bronze_path = "../delta_lake/bronze/"

dim_artists = spark.read.format("delta").load(os.path.join(bronze_path, "dim_artists"))
dim_albums = spark.read.format("delta").load(os.path.join(bronze_path, "dim_albums"))
dim_genres = spark.read.format("delta").load(os.path.join(bronze_path, "dim_genres"))
dim_tracks = spark.read.format("delta").load(os.path.join(bronze_path, "dim_tracks"))
fact_tracks = spark.read.format("delta").load(os.path.join(bronze_path, "fact_tracks"))

In [3]:
silver_path = "../delta_lake/silver/"

## Cleaning fact

In [4]:
fact_tracks.printSchema()

root
 |-- track_id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- key: string (nullable = true)
 |-- time_signature: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType, DoubleType

fact_tracks_clean = fact_tracks \
    .withColumn("popularity", col("popularity").cast(IntegerType())) \
    .withColumn("duration_ms", col("duration_ms").cast(IntegerType())) \
    .withColumn("explicit", when(col("explicit") == "True", 1)
                             .when(col("explicit") == "False", 0)
                             .otherwise(None).cast(IntegerType())) \
    .withColumn("danceability", col("danceability").cast(DoubleType())) \
    .withColumn("energy", col("energy").cast(DoubleType())) \
    .withColumn("loudness", col("loudness").cast(DoubleType())) \
    .withColumn("tempo", col("tempo").cast(DoubleType())) \
    .withColumn("speechiness", col("speechiness").cast(DoubleType())) \
    .withColumn("acousticness", col("acousticness").cast(DoubleType())) \
    .withColumn("instrumentalness", col("instrumentalness").cast(DoubleType())) \
    .withColumn("liveness", col("liveness").cast(DoubleType())) \
    .withColumn("valence", col("valence").cast(DoubleType())) \
    .withColumn("mode", col("mode").cast(IntegerType())) \
    .withColumn("key", col("key").cast(IntegerType())) \
    .withColumn("time_signature", col("time_signature").cast(IntegerType()))

In [6]:
columns_to_check = [
    "popularity", "duration_ms", "explicit",
    "danceability", "energy", "loudness", "tempo",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "mode", "key", "time_signature"
]

fact_tracks_clean = fact_tracks_clean.na.drop(subset=columns_to_check)

In [7]:
fact_tracks_clean.show(5, truncate=False)
fact_tracks_clean.printSchema()

+----------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|track_id              |popularity|duration_ms|explicit|danceability|energy|loudness|tempo  |speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|
+----------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|5SuOikwiRyPMVoIQDJUgSV|73        |230666     |0       |0.676       |0.461 |-6.746  |87.917 |0.143      |0.0322      |1.01E-6         |0.358   |0.715  |0   |1  |4             |
|4qPNDBW1i3p13qLCt0Ki3A|55        |149610     |0       |0.42        |0.166 |-17.235 |77.489 |0.0763     |0.924       |5.56E-6         |0.101   |0.267  |1   |1  |4             |
|1iJBSr7s7jYXzM8EGcbK5b|57        |210826     |0       |0.438       |0.359 |-9.734  |76.332 |0.0557     |0.21      

In [8]:
fact_tracks_clean.write.format("delta") \
    .mode("overwrite") \
    .save(os.path.join(silver_path, "fact_tracks"))

## Cleaning dim

### dim_artist

In [9]:
dim_artists.printSchema()

root
 |-- artists: string (nullable = true)
 |-- artist_id: string (nullable = true)



In [10]:
dim_artists_clean = dim_artists.withColumn("artist_id", col("artist_id").cast(IntegerType()))

dim_artists_clean = dim_artists_clean.na.drop(subset=["artists"])

In [11]:
dim_artists_clean.show(5, truncate=False)
dim_artists_clean.printSchema()

+----------------------+---------+
|artists               |artist_id|
+----------------------+---------+
|Gen Hoshino           |1        |
|Ben Woodward          |2        |
|Ingrid Michaelson;ZAYN|3        |
|Kina Grannis          |4        |
|Chord Overstreet      |5        |
+----------------------+---------+
only showing top 5 rows

root
 |-- artists: string (nullable = true)
 |-- artist_id: integer (nullable = true)



In [12]:
dim_artists_clean.write.format("delta") \
    .mode("overwrite") \
    .save(os.path.join(silver_path, "dim_artists"))

### dim_albums

In [13]:
dim_albums.printSchema()

root
 |-- album_id: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artist_id: string (nullable = true)



In [14]:

dim_albums_clean = dim_albums \
    .withColumn("album_id", col("album_id").cast(IntegerType())) \
    .withColumn("artist_id", col("artist_id").cast(IntegerType()))

dim_artists_clean = dim_albums_clean.na.drop(subset=["album_name"])

In [15]:
dim_artists_clean.show(5, truncate=False)
dim_artists_clean.printSchema()

+--------+------------------------------------------------------+---------+
|album_id|album_name                                            |artist_id|
+--------+------------------------------------------------------+---------+
|1       |Comedy                                                |1        |
|2       |Ghost (Acoustic)                                      |2        |
|3       |To Begin Again                                        |3        |
|4       |Crazy Rich Asians (Original Motion Picture Soundtrack)|4        |
|5       |Hold On                                               |5        |
+--------+------------------------------------------------------+---------+
only showing top 5 rows

root
 |-- album_id: integer (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artist_id: integer (nullable = true)



In [16]:
dim_albums.write.format("delta") \
    .mode("overwrite") \
    .save(os.path.join(silver_path, "dim_albums"))

### dim_genres

In [17]:
dim_genres.printSchema()

root
 |-- track_genre: string (nullable = true)
 |-- genre_id: string (nullable = true)



In [18]:
dim_genres_clean = dim_genres.withColumn("genre_id", col("genre_id").cast(IntegerType()))

dim_genres_clean = dim_genres_clean.na.drop(subset=["track_genre"])

In [19]:
dim_genres_clean.show(5, truncate=False)
dim_genres_clean.printSchema()

+-----------+--------+
|track_genre|genre_id|
+-----------+--------+
|acoustic   |1       |
|afrobeat   |2       |
|alt-rock   |3       |
|alternative|4       |
|ambient    |5       |
+-----------+--------+
only showing top 5 rows

root
 |-- track_genre: string (nullable = true)
 |-- genre_id: integer (nullable = true)



In [20]:
dim_genres.write.format("delta") \
    .mode("overwrite") \
    .save(os.path.join(silver_path, "dim_genres"))

### dim_tracks

In [21]:
dim_tracks.printSchema()

root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- album_id: string (nullable = true)
 |-- genre_id: string (nullable = true)



In [22]:
dim_tracks_clean = dim_tracks \
    .withColumn("album_id", col("album_id").cast(IntegerType())) \
    .withColumn("genre_id", col("genre_id").cast(IntegerType()))

dim_tracks_clean = dim_tracks_clean.na.drop(subset=["track_name"])

In [23]:
dim_tracks_clean.show(5, truncate=False)
dim_tracks_clean.printSchema()

+----------------------+--------------------------+--------+--------+
|track_id              |track_name                |album_id|genre_id|
+----------------------+--------------------------+--------+--------+
|5SuOikwiRyPMVoIQDJUgSV|Comedy                    |1       |1       |
|4qPNDBW1i3p13qLCt0Ki3A|Ghost - Acoustic          |2       |1       |
|1iJBSr7s7jYXzM8EGcbK5b|To Begin Again            |3       |1       |
|6lfxq3CG4xtTiEg7opyCyx|Can't Help Falling In Love|4       |1       |
|5vjLSffimiIP26QG5WcN2K|Hold On                   |5       |1       |
+----------------------+--------------------------+--------+--------+
only showing top 5 rows

root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- album_id: integer (nullable = true)
 |-- genre_id: integer (nullable = true)



In [24]:
dim_tracks_clean.write.format("delta") \
    .mode("overwrite") \
    .save(os.path.join(silver_path, "dim_tracks"))

## New Table

In [ ]:
fact_with_track = fact_tracks_clean.join(dim_tracks_clean, on="track_id", how="inner")

fact_with_track_selected = fact_with_track.select(
    dim_tracks_clean["track_name"],
    *[col for col in fact_tracks_clean.columns if col != "track_id"]
)

fact_with_track_selected.show(5, truncate=False)

+--------------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|track_name                |popularity|duration_ms|explicit|danceability|energy|loudness|tempo  |speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|
+--------------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|Comedy                    |73        |230666     |0       |0.676       |0.461 |-6.746  |87.917 |0.143      |0.0322      |1.01E-6         |0.358   |0.715  |0   |1  |4             |
|Ghost - Acoustic          |55        |149610     |0       |0.42        |0.166 |-17.235 |77.489 |0.0763     |0.924       |5.56E-6         |0.101   |0.267  |1   |1  |4             |
|To Begin Again            |57        |210826     |0       |0.438       |0.359 |-9.734  |76.332